In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit
import statsmodels.api as sm

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
print('Libraries loaded.')

Libraries loaded.


In [67]:
# Load all data sources
consol = pd.read_csv('consolidated_db.csv', parse_dates=['trade_date'])
dam = pd.read_csv('uploads/da_model_dataset.csv', parse_dates=['date'])
eua = pd.read_csv('uploads/eua_front_year.csv', parse_dates=['trade_date'])

dam.rename(columns={'date': 'trade_date'}, inplace=True)

# Get columns from da_model_dataset that aren't already in consolidated
dam_cols = [c for c in dam.columns if c not in consol.columns and c != 'trade_date']

# Merge everything
master = consol.merge(dam[['trade_date'] + dam_cols], on='trade_date', how='left')
master = master.merge(
    eua[['trade_date', 'volume', 'open_interest']].rename(
        columns={'volume': 'eua_volume', 'open_interest': 'eua_oi'}),
    on='trade_date', how='left'
)

print(f'Master dataset: {master.shape[0]} trading days × {master.shape[1]} columns')
print(f'Date range: {master.trade_date.min().date()} to {master.trade_date.max().date()}')
print(f'\nColumns from consolidated_db: {len(consol.columns)}')
print(f'New columns from da_model_dataset: {len(dam_cols)}')
print(f'New columns from eua_front_year: 2 (volume, open interest)')

Master dataset: 1310 trading days × 70 columns
Date range: 2021-01-04 to 2026-02-25

Columns from consolidated_db: 22
New columns from da_model_dataset: 46
New columns from eua_front_year: 2 (volume, open interest)


## Existing Linear Features

Existing linear features
- **Energy base**: lagged returns of EUA, TTF, Coal, Brent, Power, and spread changes (CDS, CSS)
- **Auctions**: cover ratio, auction premium, bidder deviations, excess demand events
- **COT positions**: z-scores, net changes, concentration measures for each actor group
- **Options**: ATM implied volatility, risk reversals, butterfly spreads, VRP, OI changes

All of these are already lagged by 1 day (shift(1))

In [68]:
# Features already in da_model_dataset (pre-lagged)
dam_features = [c for c in dam_cols if c != 'lag_cot_date' and c != 'lag_r_EUA']

# Base energy features from consolidated_db
energy_base = ['lag_r_EUA', 'lag_r_TTF', 'lag_r_Coal', 'lag_r_Brent', 
               'lag_r_Power', 'lag_d_CDS', 'lag_d_CSS']

# Combine into full linear candidate set (no duplicates)
linear_candidates = list(dict.fromkeys(energy_base + dam_features))
linear_candidates = [c for c in linear_candidates if c in master.columns]

# Categorise them
auction_lin = [c for c in dam_features if any(x in c for x in ['auction','cover','bidder','excess'])]
cot_lin = [c for c in dam_features if any(x in c for x in ['IF_','CU_','IFCI_','OWCO_','cot','spec','concentration'])]
opts_lin = [c for c in dam_features if any(x in c.lower() for x in ['iv','oi','rr_','bf_','vrp','rv_','dec_']) 
            and c not in cot_lin]

print(f'Total linear features: {len(linear_candidates)}')
print(f'  Energy base: {len(energy_base)}')
print(f'  Auction: {len(auction_lin)}')
print(f'  COT: {len(cot_lin)}')
print(f'  Options: {len(opts_lin)}')
print(f'  Other/overlap: {len(linear_candidates) - len(energy_base) - len(auction_lin) - len(cot_lin) - len(opts_lin)}')

Total linear features: 52
  Energy base: 7
  Auction: 6
  COT: 23
  Options: 17
  Other/overlap: -1


## Non-Linear Feature Engineering

### Energy Non-Linear Features

three types of energy transforms:

**Squared returns** capture volatility effects — large moves (positive or negative) in gas/coal prices may affect EUA more than small moves, regardless of direction

**Price ratios** capture fuel-switching incentives — when coal becomes cheap relative to gas, power generators switch to coal (more emissions, higher EUA demand)

**Spread levels** capture the economics of power generation

In [69]:
# Energy non-linear features
energy_nl = []

# a. Squared returns (volatility proxy)
for col in ['lag_r_TTF', 'lag_r_Coal', 'lag_r_Brent', 'lag_r_Power']:
    if col in master.columns:
        master[f'{col}_sq'] = master[col] ** 2
        energy_nl.append(f'{col}_sq')

# b. Price ratios (fuel switching)
if all(c in master.columns for c in ['Coal', 'TTF']):
    master['lag_coal_gas_ratio'] = (master['Coal'] / master['TTF'].replace(0, np.nan)).shift(1)
    energy_nl.append('lag_coal_gas_ratio')

# c. Spread levels
if all(c in master.columns for c in ['Power', 'TTF', 'Coal']):
    master['lag_spark_spread'] = (master['Power'] - master['TTF']).shift(1)
    master['lag_dark_spread'] = (master['Power'] - master['Coal']).shift(1)
    energy_nl.extend(['lag_spark_spread', 'lag_dark_spread'])

print(f'Block 1 — Energy non-linear features: {len(energy_nl)}')
for f in energy_nl:
    print(f'  {f}: {master[f].notna().mean():.1%} coverage')

Block 1 — Energy non-linear features: 7
  lag_r_TTF_sq: 99.8% coverage
  lag_r_Coal_sq: 99.8% coverage
  lag_r_Brent_sq: 99.8% coverage
  lag_r_Power_sq: 99.8% coverage
  lag_coal_gas_ratio: 99.9% coverage
  lag_spark_spread: 99.9% coverage
  lag_dark_spread: 99.9% coverage


### Auction Non-Linear Features

**Squared auction premium** captures whether extreme auction outcomes (very high or very low premiums) matter more than moderate ones

**Squared cover ratio** tests whether extremely over- or under-subscribed auctions have outsized effects

**Low cover regime** flags auctions with historically weak demand (below 25th percentile), which may signal supply pressure.

In [70]:
# Auction non-linear features
auction_nl = []

for col in ['lag_eu_auction_premium', 'lag_eu_cover_ffill_surp']:
    if col in master.columns:
        master[f'{col}_sq'] = master[col] ** 2
        auction_nl.append(f'{col}_sq')

print(f'Block 2 — Auction non-linear features: {len(auction_nl)}')
for f in auction_nl:
    print(f'  {f}: {master[f].notna().mean():.1%} coverage')

Block 2 — Auction non-linear features: 2
  lag_eu_auction_premium_sq: 99.8% coverage
  lag_eu_cover_ffill_surp_sq: 98.3% coverage


### COT Non-Linear Features

**Squared z-scores** capture whether extreme positioning (either direction) matters

**Speculator–commercial imbalance** measures the gap between investment fund and commercial positioning. When speculators are heavily long relative to commercials, the market may be vulnerable to a reversal

**Crowded positioning indicator** flags when speculator positioning exceeds ±1.5 standard deviations from its 26-week mean — a potential contrarian signal.

In [71]:
# COT non-linear features
cot_nl = []

for col in ['lag_IF_net_z26', 'lag_CU_net_z26', 'lag_IF_concentration']:
    if col in master.columns:
        master[f'{col}_sq'] = master[col] ** 2
        cot_nl.append(f'{col}_sq')

# Spec-commercial imbalance
if all(c in master.columns for c in ['lag_IF_net_z26', 'lag_CU_net_z26']):
    master['lag_spec_comm_imbalance'] = master['lag_IF_net_z26'] - master['lag_CU_net_z26']
    cot_nl.append('lag_spec_comm_imbalance')

# Crowded positioning flag
if 'lag_IF_net_z26' in master.columns:
    master['lag_crowded_spec'] = (master['lag_IF_net_z26'].abs() > 1.5).astype(int)
    cot_nl.append('lag_crowded_spec')

print(f'Block 3 — COT non-linear features: {len(cot_nl)}')
for f in cot_nl:
    print(f'  {f}: {master[f].notna().mean():.1%} coverage')

Block 3 — COT non-linear features: 5
  lag_IF_net_z26_sq: 96.2% coverage
  lag_CU_net_z26_sq: 96.2% coverage
  lag_IF_concentration_sq: 99.2% coverage
  lag_spec_comm_imbalance: 96.2% coverage
  lag_crowded_spec: 100.0% coverage


### Options Non-Linear Features

**Squared IV and Greeks** test whether extreme volatility conditions have outsized effects

**High-IV regime indicator** flags periods when implied volatility is above its 252-day 75th percentile. Options pricing during these regimes may contain stronger directional signals.

In [72]:
# Options non-linear features
opts_nl = []

for col in ['lag_atm_iv', 'lag_rr_25', 'lag_vrp', 'lag_rv_ratio']:
    if col in master.columns:
        master[f'{col}_sq'] = master[col] ** 2
        opts_nl.append(f'{col}_sq')

# High-IV regime
if 'lag_atm_iv' in master.columns:
    iv_p75 = master['lag_atm_iv'].rolling(252, min_periods=60).quantile(0.75)
    master['lag_high_iv_regime'] = (master['lag_atm_iv'] > iv_p75).astype(int)
    opts_nl.append('lag_high_iv_regime')

print(f'Block 4 — Options non-linear features: {len(opts_nl)}')
for f in opts_nl:
    print(f'  {f}: {master[f].notna().mean():.1%} coverage')

Block 4 — Options non-linear features: 5
  lag_atm_iv_sq: 91.6% coverage
  lag_rr_25_sq: 91.6% coverage
  lag_vrp_sq: 91.6% coverage
  lag_rv_ratio_sq: 99.8% coverage
  lag_high_iv_regime: 100.0% coverage


### Cross-Block Interactions

test whether the effect of one variable depends on the level of another from a different data source. For example:
- **Energy × COT**: Do gas price moves matter more when speculator positioning is stretched?  
  
- **Energy × Options**: Do energy variables matter more when option-implied uncertainty is high?  
 
- **Auction × COT**: Does auction pressure interact with speculator positioning?

- **COT × Options**: Does positioning interact with volatility expectations?


In [73]:
# Cross-block interactions
cross_nl = []

interaction_pairs = [
    ('lag_r_TTF', 'lag_IF_net_z26', 'lag_TTF_x_IF_z26'),       # energy × COT
    ('lag_r_TTF', 'lag_atm_iv', 'lag_TTF_x_IV'),               # energy × options
    ('lag_r_Coal', 'lag_CU_net_z26', 'lag_Coal_x_CU_z26'),     # energy × COT
    ('lag_eu_auction_premium', 'lag_IF_net_z26', 'lag_auc_x_IF'),  # auction × COT
    ('lag_IF_net_z26', 'lag_atm_iv', 'lag_IF_x_IV'),           # COT × options
]

for a, b, name in interaction_pairs:
    if all(c in master.columns for c in [a, b]):
        master[name] = master[a] * master[b]
        cross_nl.append(name)

print(f'Block 5 — Cross-block interactions: {len(cross_nl)}')
for f in cross_nl:
    print(f'  {f}: {master[f].notna().mean():.1%} coverage')

Block 5 — Cross-block interactions: 5
  lag_TTF_x_IF_z26: 96.2% coverage
  lag_TTF_x_IV: 91.6% coverage
  lag_Coal_x_CU_z26: 96.2% coverage
  lag_auc_x_IF: 96.2% coverage
  lag_IF_x_IV: 91.6% coverage


In [74]:
# Combine all non-linear features
all_nl_features = energy_nl + auction_nl + cot_nl + opts_nl + cross_nl

print(f'{"="*60}')
print(f'FEATURE ENGINEERING SUMMARY')
print(f'{"="*60}')
print(f'Linear features:     {len(linear_candidates)}')
print(f'Non-linear features (new):           {len(all_nl_features)}')
print(f'  Block 1 — Energy NL:               {len(energy_nl)}')
print(f'  Block 2 — Auction NL:              {len(auction_nl)}')
print(f'  Block 3 — COT NL:                  {len(cot_nl)}')
print(f'  Block 4 — Options NL:              {len(opts_nl)}')
print(f'  Block 5 — Cross-block:             {len(cross_nl)}')
print(f'{"="*60}')

# Full candidate set
all_candidates = list(dict.fromkeys(linear_candidates + all_nl_features))
all_candidates = [c for c in all_candidates if c in master.columns]
print(f'\nTotal candidate pool: {len(all_candidates)} features')

FEATURE ENGINEERING SUMMARY
Linear features:     52
Non-linear features (new):           24
  Block 1 — Energy NL:               7
  Block 2 — Auction NL:              2
  Block 3 — COT NL:                  5
  Block 4 — Options NL:              5
  Block 5 — Cross-block:             5

Total candidate pool: 76 features



### StandardScaler
Before penalised regression, we standardise all features to mean=0, std=1. This is essential because:
- Lasso/Elastic Net penalise the **magnitude** of coefficients
- Without scaling, features with larger natural scales (e.g., open interest in millions) get penalised more heavily than features with small scales (e.g., z-scores around ±2)
- StandardScaler ensures the penalty treats all features equally

We fit the scaler on training data only and apply it to test data (no data leakage).

In [75]:
# Replace infinities (from division by zero in ratios)
for c in all_candidates:
    master[c] = master[c].replace([np.inf, -np.inf], np.nan)

# Check coverage
target = 'r_EUA'
coverage = master[all_candidates].notna().mean()

print('Feature coverage distribution:')
print(f'  100% coverage:  {(coverage == 1.0).sum()} features')
print(f'  80-99% coverage: {((coverage >= 0.80) & (coverage < 1.0)).sum()} features')
print(f'  70-79% coverage: {((coverage >= 0.70) & (coverage < 0.80)).sum()} features')
print(f'  <70% coverage:  {(coverage < 0.70).sum()} features')

# Apply 70% threshold
good_candidates = [c for c in all_candidates if coverage[c] >= 0.70]
dropped = [c for c in all_candidates if coverage[c] < 0.70]

if dropped:
    print(f'\nDropped {len(dropped)} features with <70% coverage:')
    for c in dropped:
        print(f'  {c}: {coverage[c]:.1%}')

all_candidates = good_candidates
print(f'\nFinal candidate set: {len(all_candidates)} features')

Feature coverage distribution:
  100% coverage:  2 features
  80-99% coverage: 67 features
  70-79% coverage: 0 features
  <70% coverage:  7 features

Dropped 7 features with <70% coverage:
  lag_d_IF_eex_net: 62.8%
  lag_TTF_IF_net_zscore: 61.6%
  lag_d_TTF_IF_net_m: 62.7%
  lag_d_TTF_IF_net_pct: 62.7%
  lag_TTF_IF_net_pct: 63.1%
  lag_d_TTF_IFCI_net_pct: 62.7%
  lag_TTF_CU_spec_net: 63.1%

Final candidate set: 69 features


In [76]:
# Build model matrix (drop any remaining NaN rows)
model_data = master[[target, 'trade_date'] + all_candidates].dropna().reset_index(drop=True)

print(f'Model data: {model_data.shape[0]} rows')
print(f'Date range: {model_data.trade_date.min().date()} to {model_data.trade_date.max().date()}')

# Chronological 80/20 split
n = len(model_data)
split_idx = int(n * 0.80)

train_data = model_data.iloc[:split_idx].copy()
test_data = model_data.iloc[split_idx:].copy()

print(f'\nTrain: {len(train_data)} rows ({train_data.trade_date.min().date()} to {train_data.trade_date.max().date()})')
print(f'Test:  {len(test_data)} rows ({test_data.trade_date.min().date()} to {test_data.trade_date.max().date()})')

# Extract X and y
X_train = train_data[all_candidates].values
X_test = test_data[all_candidates].values
y_train = train_data[target].values
y_test = test_data[target].values

# Standardise
scaler = StandardScaler()
X_train_scaled = np.nan_to_num(scaler.fit_transform(X_train), nan=0, posinf=0, neginf=0)
X_test_scaled = np.nan_to_num(scaler.transform(X_test), nan=0, posinf=0, neginf=0)

print(f'\nFeature matrix: {X_train_scaled.shape[0]} train × {X_train_scaled.shape[1]} features')

Model data: 963 rows
Date range: 2021-06-09 to 2026-02-25

Train: 770 rows (2021-06-09 to 2025-01-07)
Test:  193 rows (2025-01-08 to 2026-02-25)

Feature matrix: 770 train × 69 features


## Lasso Regression (L1 Penalty)

### How Lasso works
Lasso minimises:

The L1 penalty ($|\beta_j|$) has a special geometric property: it drives coefficients to **exactly zero**. This means Lasso performs automatic feature selection — features that don't contribute enough predictive power are removed entirely.

### Choosing alpha
We use `LassoCV` with `TimeSeriesSplit` cross-validation (5 folds, expanding window). This respects the temporal ordering of our data — the model is always validated on data that comes after the training data, just like in real forecasting.

The alpha grid spans from $10^{-7}$ (very weak penalty, keeps many features) to $10^{-1}$ (strong penalty, removes most features).

In [77]:
# Alpha grid: from weak penalty to strong penalty
alphas = np.logspace(-7, -1, 100)

# Time-series cross-validation (5 expanding folds)
tscv = TimeSeriesSplit(n_splits=5)

# Fit LassoCV
lasso_cv = LassoCV(alphas=alphas, cv=tscv, max_iter=50000, tol=1e-4, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

# results
lasso_coefs = pd.Series(lasso_cv.coef_, index=all_candidates)
lasso_survivors = lasso_coefs[lasso_coefs != 0].sort_values(key=abs, ascending=False)

print(f'Optimal alpha (Lasso): {lasso_cv.alpha_:.8f}')
print(f'Features surviving Lasso: {len(lasso_survivors)} / {len(all_candidates)}')
print()

if len(lasso_survivors) > 0:
    print('Surviving features (sorted by |coefficient|):')
    for feat, coef in lasso_survivors.items():
        feat_type = 'NL' if feat in set(all_nl_features) else 'LIN'
        print(f'  [{feat_type}] {feat:45s} {coef:+.10f}')
else:
    print('Lasso zeroed ALL features.')
    print('The optimal penalty is so strong that no feature improves CV performance.')

Optimal alpha (Lasso): 0.10000000
Features surviving Lasso: 0 / 69

Lasso zeroed ALL features.
The optimal penalty is so strong that no feature improves CV performance.


## Elastic Net (L1 + L2 Penalty)

### Why Elastic Net?
Lasso has a limitation: when features are correlated, it tends to pick one and zero the others arbitrarily. Elastic Net adds an L2 (Ridge) penalty that keeps correlated features together

where $\rho$ (`l1_ratio`) controls the mix:
- $\rho = 1$: pure Lasso
- $\rho = 0$: pure Ridge (no feature selection)
- $\rho = 0.5$: equal mix

We search over multiple l1_ratio values to find the best combination.

In [78]:
# Elastic Net with multiple l1_ratio values
enet_cv = ElasticNetCV(
    alphas=alphas,
    l1_ratio=[0.1, 0.5, 0.9, 0.99],
    cv=tscv,
    max_iter=50000,
    tol=1e-4,
    random_state=42
)
enet_cv.fit(X_train_scaled, y_train)

# Results
enet_coefs = pd.Series(enet_cv.coef_, index=all_candidates)
enet_survivors = enet_coefs[enet_coefs != 0].sort_values(key=abs, ascending=False)

print(f'Optimal alpha (Elastic Net): {enet_cv.alpha_:.8f}')
print(f'Optimal l1_ratio: {enet_cv.l1_ratio_:.2f}')
print(f'Features surviving Elastic Net: {len(enet_survivors)} / {len(all_candidates)}')
print()

if len(enet_survivors) > 0:
    print('Surviving features:')
    for feat, coef in enet_survivors.head(20).items():
        feat_type = 'NL' if feat in set(all_nl_features) else 'LIN'
        print(f'  [{feat_type}] {feat:45s} {coef:+.10f}')
else:
    print('Elastic Net zeroed ALL features.')

Optimal alpha (Elastic Net): 0.10000000
Optimal l1_ratio: 0.50
Features surviving Elastic Net: 0 / 69

Elastic Net zeroed ALL features.


## OLS with Penalised Survivors — Win Rate Evaluation
We compute baselines:
- **AR(1)**: Only lagged EUA return — the simplest possible model
- **Kitchen sink**: All features, no penalty — shows the overfitting risk

In [79]:
# AR(1) baseline
ar_idx = all_candidates.index('lag_r_EUA')
X_ar_train = sm.add_constant(X_train_scaled[:, ar_idx:ar_idx+1])
X_ar_test = sm.add_constant(X_test_scaled[:, ar_idx:ar_idx+1])
ols_ar = sm.OLS(y_train, X_ar_train).fit()
pred_ar = ols_ar.predict(X_ar_test)
wr_ar = ((pred_ar > 0) == (y_test > 0)).mean()
print(f'\nAR(1) baseline:')
print(f'  Test Win Rate: {wr_ar:.2%}')

# Kitchen sink (all features, no penalty)
X_all_train = sm.add_constant(X_train_scaled)
X_all_test = sm.add_constant(X_test_scaled)
ols_all = sm.OLS(y_train, X_all_train).fit()
pred_all = ols_all.predict(X_all_test)
wr_all = ((pred_all > 0) == (y_test > 0)).mean()
print(f'\nKitchen sink (all {len(all_candidates)} features, no penalty):')
print(f'  Test Win Rate: {wr_all:.2%}')
print(f'  Train R²: {ols_all.rsquared:.6f}')


AR(1) baseline:
  Test Win Rate: 51.81%

Kitchen sink (all 69 features, no penalty):
  Test Win Rate: 47.67%
  Train R²: 0.189367


In [82]:
print('='*70)
print('NON-LINEAR FEATURES RESULTS')
print('='*70)
print()
print(f'Feature engineering:')
print(f'  Linear features :    {len(linear_candidates)}')
print(f'  Non-linear features (new):    {len(all_nl_features)}')
print(f'  Total candidates:             {len(all_candidates)}')
print(f'  Sample size:                  {len(model_data)} days')
print(f'  Train/Test split:             {len(train_data)}/{len(test_data)}')
print()
print(f'Penalised selection:')
print(f'  Lasso survivors:              {len(lasso_survivors)}/{len(all_candidates)}')
print(f'  Elastic Net survivors:        {len(enet_survivors)}/{len(all_candidates)}')
print()
print(f'{"Model":<45} {"Test WR":<10} {"Notes"}')
print(f'{"-"*70}')
print(f'{"DA_0: AR(1) ":<45} {wr_ar:.2%}{"":>5} {"Simplest baseline"}')
print(f'{"Kitchen sink — no penalty":<45} {wr_all:.2%}{"":>5} {f"All {len(all_candidates)} features"}')


NON-LINEAR FEATURES RESULTS

Feature engineering:
  Linear features :    52
  Non-linear features (new):    24
  Total candidates:             69
  Sample size:                  963 days
  Train/Test split:             770/193

Penalised selection:
  Lasso survivors:              0/69
  Elastic Net survivors:        0/69

Model                                         Test WR    Notes
----------------------------------------------------------------------
DA_0: AR(1)                                   51.81%      Simplest baseline
Kitchen sink — no penalty                     47.67%      All 69 features
